In [ ]:
# notebook 09: to make PPI network diagrams, for targets that cluster similarly

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import fisher_exact, spearmanr
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from itertools import combinations

from adjustText import adjust_text
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

import gseapy as gp
import time

# load data saved from notebook 08

# target-cluster long-form odds_ratio table (with padj etc) from Fisher's exact test, used for heatmap
enrichment = pd.read_csv("../data/interim/08_perturbation_cluster_enrichment.csv.gz")
# target x cluster wide form: values = fraction of target cells in each clusterName1 (raw input before Fisher's)
composition = pd.read_csv("../data/interim/08_cluster_composition.csv.gz", index_col=0)
# AnnData for L2FC matrix, after z-scoring, with PCA/UMAP/Leiden
pert_adata = sc.read_h5ad("../data/interim/08_pert_adata_manifold.h5ad")

adata = sc.read_h5ad("../data/interim/07_adata_clustered.h5ad")  # only if 09 needs cell-level data too

In [ ]:
# query STRING-db for interactions among one cluster's genes.  using REST API.
# network enpoint --> confidence scored interaction

import requests
import networkx as nx

STRING_API_URL = "https://string-db.org/api"

def get_string_interactions(genes, species=9606, score_threshold=400):
    params = {
        "identifiers": "\r".join(genes),  # actual CR character, not the literal string "%0d"
        "species": species,
        "required_score": score_threshold,
    }
    response = requests.get(f"{STRING_API_URL}/tsv/network", params=params)
    response.raise_for_status()
    lines = response.text.strip().split("\n")
    header = lines[0].split("\t")
    rows = [line.split("\t") for line in lines[1:]]
    return pd.DataFrame(rows, columns=header)

def get_string_actions(genes, species=9606, score_threshold=400):
    """Interaction TYPE (activation/inhibition/phosphorylation/binding/etc.) and directionality where known."""
    params = {
        "identifiers": "%0d".join(genes),
        "species": species,
        "required_score": score_threshold,
    }
    response = requests.get(f"{STRING_API_URL}/tsv/actions", params=params)
    response.raise_for_status()
    
    lines = response.text.strip().split("\n")
    header = lines[0].split("\t")
    rows = [line.split("\t") for line in lines[1:]]
    return pd.DataFrame(rows, columns=header)

In [ ]:
# ── Loop through every manifold cluster, query STRING for interactions among its member genes ──
import time

cluster_interactions = {}
for cluster_id in pert_adata.obs["leiden"].cat.categories:
    genes = pert_adata.obs[pert_adata.obs["leiden"] == cluster_id].index.tolist()
    try:
        cluster_interactions[cluster_id] = get_string_interactions(genes)
    except Exception as e:
        print(f"cluster {cluster_id} failed: {e}")
        cluster_interactions[cluster_id] = pd.DataFrame()
    time.sleep(1)  # be polite to STRING's API, same reasoning as the Enrichr rate limiting earlier

for cluster_id, df in cluster_interactions.items():
    print(f"Cluster {cluster_id}: {len(df)} interactions")

In [ ]:
import networkx as nx
from matplotlib.lines import Line2D

palette = sc.pl.palettes.default_20
cluster_colors = dict(zip(pert_adata.obs["leiden"].cat.categories, palette))

n_clusters = len(cluster_interactions)
n_cols = 3
n_rows = int(np.ceil(n_clusters / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = axes.flatten()

for i, (cluster_id, edges_df) in enumerate(cluster_interactions.items()):
    ax = axes[i]
    genes = pert_adata.obs[pert_adata.obs["leiden"] == cluster_id].index.tolist()

    G = nx.Graph()
    G.add_nodes_from(genes)

    if len(edges_df) > 0:
        for _, row in edges_df.iterrows():
            G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=float(row["score"]))

    pos = nx.spring_layout(G, seed=42, k=0.8)

    node_color = cluster_colors[cluster_id]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_color, node_size=500, edgecolors="black", linewidths=0.8)

    if G.number_of_edges() > 0:
        weights = [G[u][v]["weight"] * 3 for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, ax=ax, width=weights, edge_color="gray", alpha=0.7)

    nx.draw_networkx_labels(
        G, pos, ax=ax, font_size=10, font_weight="bold", font_color="white",
        bbox=dict(boxstyle="round,pad=0.2", facecolor=node_color, edgecolor="none", alpha=1.0),
    )

    ax.set_title(f"cluster {cluster_id}:({G.number_of_nodes()} targets, of which {G.number_of_edges()} have STRING interactions)",
                 pad=0)
    ax.axis("off")

for j in range(n_clusters, len(axes)):
    axes[j].axis("off")

fig.suptitle("STRING PPI networks within each manifold cluster", fontsize=20, y=1.02)
fig.tight_layout()

# ── Thin grey divider lines between panels, computed from actual subplot positions after layout ──
fig.canvas.draw()

row_positions = [ax.get_position() for ax in axes[:n_clusters]]
grid_top = max(p.y1 for p in row_positions)
grid_bottom = min(p.y0 for p in row_positions)

# ── Vertical dividers only, each split into two segments (top row span, bottom row span) with a gap between ──
gap_fraction = 0.03  # fraction of total figure height to leave blank at the row boundary

# y-boundary between row 0 and row 1, same midpoint logic as before
row0_bottom = axes[0].get_position().y0
row1_top = axes[n_cols].get_position().y1
row_mid = (row0_bottom + row1_top) / 2

for col in range(n_cols - 1):
    x_right = axes[col].get_position().x1
    x_left = axes[col + 1].get_position().x0
    x_mid = (x_right + x_left) / 2

    # top segment: from top of grid down to just above the row midpoint
    line_top = Line2D([x_mid, x_mid], [row_mid + gap_fraction / 2, grid_top], transform=fig.transFigure,
                       color="lightgrey", linewidth=1)
    fig.add_artist(line_top)

    # bottom segment: from just below the row midpoint down to bottom of grid
    line_bottom = Line2D([x_mid, x_mid], [grid_bottom, row_mid - gap_fraction / 2], transform=fig.transFigure,
                          color="lightgrey", linewidth=1)
    fig.add_artist(line_bottom)

fig.savefig("../results/figures/manifold_cluster_STRING_networks.png", dpi=200, bbox_inches="tight")
plt.show()